In [ ]:

import uuid, time
from datetime import datetime, timezone
from pyspark.sql import Row, functions as F
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DoubleType, IntegerType
)

import sys
dbutils.widgets.text("shared_lib_path", "/Workspace/Shared")
sys.path.insert(0, dbutils.widgets.get("shared_lib_path"))
sys.path.append("/Workspace/Shared")   # fallback: files not yet deployed to bundle path
from pipeline_logging import pipeline_log_upsert, pipeline_step_log_upsert, ingestion_log_insert
import pipeline_utils as Utils


# ---------------------------------------------------------------------------
# catalog is injected as a job parameter by the bundle (${var.catalog}).
# Falls back to "vinoworld" for manual/standalone notebook runs.
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "vinoworld")
CATALOG   = dbutils.widgets.get("catalog")
BRONZE    = f"{CATALOG}.bronze"
SILVER    = f"{CATALOG}.silver"
GOLD      = f"{CATALOG}.gold"
AUDIT     = f"{CATALOG}.audit"
RAW_FILES = f"/Volumes/{CATALOG}/datafiles/"

STATUS_RUNNING   = "running"
STATUS_SUCCEEDED = "succeeded"
STATUS_FAILED    = "failed"
STATUS_NO_FILES  = "no_files"


# --------------------------------------------------------------------------
# Get pipeline parameters for pipeline_log table info.
# --------------------------------------------------------------------------

LOCAL_PIPELINE_ID = "11111"    # fallback when run manually outside a job


# Pipeline-level identifiers — shared across all notebooks in a pipeline run.
# Read from parent via widgets; fall back to standalone mode if not provided.
dbutils.widgets.text("pipeline_run_id", "")
value = dbutils.widgets.get("pipeline_run_id")

if not value:
    # Running as a Job task — try to get it from the init task's taskValues
    try:
        value = str(dbutils.jobs.taskValues.get(
            taskKey    = "init_pipeline_log",
            key        = "pipeline_run_id",
            debugValue = LOCAL_PIPELINE_ID
        ))
    except Exception:
        value = str(LOCAL_PIPELINE_ID)

PIPELINE_RUN_ID = value
